# Week 3 — CT (Radon / FBP) and MRI (k-space), hands-on

This notebook makes the two forward operators from the theory concrete:
- **CT:** \(G\) = **Radon transform**. We build a sinogram, reconstruct with plain
  vs **filtered** back-projection, and watch quality fall as we drop view angles.
- **MRI:** \(G\) = **Fourier transform**. We under-sample **k-space** and watch
  aliasing appear.

Both are just the reconstruction template \(y=Gx+n\) with a specific \(G\); the
inversion maths is in the `week_1.5_to_3` deblurring notebook. NumPy + Matplotlib
only — every operator is `np.fft` plus a little interpolation, nothing hidden.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
plt.rcParams['figure.figsize'] = (10, 4)

def head_phantom(N=96):
    yy, xx = np.mgrid[0:N, 0:N].astype(float)
    c = (N - 1) / 2.0
    X = (xx - c) / (N / 2.0); Y = (yy - c) / (N / 2.0)
    img = np.zeros((N, N))
    img[(X/0.92)**2 + (Y/0.92)**2 <= 1] = 0.3
    img[(X/0.80)**2 + (Y/0.80)**2 <= 1] = 0.6
    img[((X+0.25)/0.18)**2 + ((Y-0.10)/0.18)**2 <= 1] = 0.95
    img[((X-0.30)/0.12)**2 + ((Y+0.20)/0.22)**2 <= 1] = 0.10
    return img

def psnr(ref, test, peak=1.0):
    return 10*np.log10(peak**2 / np.mean((ref - test)**2))

clean = head_phantom()
plt.imshow(clean, cmap='gray', vmin=0, vmax=1)
plt.title('attenuation phantom (the true image x)'); plt.colorbar(); plt.axis('off'); plt.show()

## CT step 1 — the Radon transform (build the sinogram)

Each X-ray measures a line integral of attenuation; collecting them over all
angles is the **Radon transform**, displayed as the **sinogram**. We compute it
by rotating the image and summing along columns (a line integral at each angle).

In [ ]:
def rotate(img, angle_deg):
    """Bilinear image rotation about the centre (numpy only)."""
    M = img.shape[0]
    c = (M - 1) / 2.0
    th = np.deg2rad(angle_deg)
    cos, sin = np.cos(th), np.sin(th)
    yy, xx = np.mgrid[0:M, 0:M].astype(float)
    xr, yr = xx - c, yy - c
    xs = cos*xr + sin*yr + c
    ys = -sin*xr + cos*yr + c
    x0 = np.floor(xs).astype(int); y0 = np.floor(ys).astype(int)
    wx, wy = xs - x0, ys - y0
    out = np.zeros((M, M))
    for dy in (0, 1):
        for dx in (0, 1):
            xi, yi = x0 + dx, y0 + dy
            w = (wx if dx else 1 - wx) * (wy if dy else 1 - wy)
            ok = (xi >= 0) & (xi < M) & (yi >= 0) & (yi < M)
            out += np.where(ok, img[np.clip(yi, 0, M-1), np.clip(xi, 0, M-1)] * w, 0.0)
    return out

def radon(img, thetas):
    sino = np.zeros((img.shape[0], len(thetas)))
    for k, t in enumerate(thetas):
        sino[:, k] = rotate(img, t).sum(axis=0)
    return sino

thetas = np.linspace(0.0, 180.0, 90, endpoint=False)
sino = radon(clean, thetas)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(clean, cmap='gray'); ax[0].set_title('image'); ax[0].axis('off')
ax[1].imshow(sino, cmap='gray', aspect='auto', extent=[0, 180, 0, sino.shape[0]])
ax[1].set_title('sinogram = Radon transform (the CT data y)')
ax[1].set_xlabel('angle (deg)'); ax[1].set_ylabel('detector position')
plt.tight_layout(); plt.show()

## CT step 2 — back-projection vs filtered back-projection

Plain back-projection (smear each projection back along its ray and sum)
**blurs**. Multiplying each projection's spectrum by the **|w| ramp** before
back-projecting fixes it — that is **FBP**. Then we strip away view angles and
watch the streaks appear (exactly where iterative/MAP reconstruction earns its keep).

In [ ]:
def back_project(sino, thetas, ramp=True):
    M = sino.shape[0]
    recon = np.zeros((M, M))
    ramp_filter = np.abs(np.fft.fftfreq(M))          # |w|  (Ram-Lak)
    for k, t in enumerate(thetas):
        p = sino[:, k]
        if ramp:
            p = np.real(np.fft.ifft(np.fft.fft(p) * ramp_filter))
        band = np.tile(p, (M, 1))                     # smear projection...
        recon += rotate(band, -t)                     # ...back along its angle
    return recon * (np.pi / len(thetas))

bp_plain = back_project(sino, thetas, ramp=False)
fbp_full = back_project(sino, thetas, ramp=True)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(bp_plain, cmap='gray'); ax[0].set_title('plain back-projection (blurs)'); ax[0].axis('off')
ax[1].imshow(fbp_full, cmap='gray'); ax[1].set_title('FBP, 90 angles (sharp)'); ax[1].axis('off')
ax[2].imshow(clean, cmap='gray'); ax[2].set_title('true x'); ax[2].axis('off')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a_, nviews in zip(ax, [60, 20, 8]):
    th = np.linspace(0, 180, nviews, endpoint=False)
    rec = back_project(radon(clean, th), th, ramp=True)
    a_.imshow(rec, cmap='gray'); a_.set_title(f'FBP, {nviews} angles'); a_.axis('off')
plt.tight_layout(); plt.show()

## MRI — k-space and under-sampling

MRI measures the **Fourier transform** of the image (k-space); reconstruction is
an inverse FFT (take the magnitude). Skipping k-space lines speeds the scan but
injects **aliasing/blur** — the inverse problem compressed sensing later solves
with a sparsity prior. Below: full k-space, centre-only (low-pass), and a random
subset.

In [ ]:
kspace = np.fft.fftshift(np.fft.fft2(clean))
M = clean.shape[0]
cy, cx = M // 2, M // 2

def recon_from_kspace(mask):
    return np.abs(np.fft.ifft2(np.fft.ifftshift(kspace * mask)))

full_mask = np.ones((M, M))
low_mask = np.zeros((M, M)); r = M // 8
low_mask[cy-2*r:cy+2*r, cx-2*r:cx+2*r] = 1.0
rand_mask = (rng.random((M, M)) < 0.30).astype(float)
rand_mask[cy-r:cy+r, cx-r:cx+r] = 1.0            # always keep the centre (contrast)

masks = [('full k-space', full_mask),
         ('centre 25% only (low-pass blur)', low_mask),
         ('random 30% (aliasing)', rand_mask)]
fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for j, (name, m) in enumerate(masks):
    ax[0, j].imshow(np.log1p(np.abs(kspace * m)), cmap='gray'); ax[0, j].axis('off')
    ax[0, j].set_title('k-space: ' + name)
    rec = recon_from_kspace(m)
    ax[1, j].imshow(rec, cmap='gray', vmin=0, vmax=1); ax[1, j].axis('off')
    ax[1, j].set_title(f'recon  (PSNR {psnr(clean, rec):.1f} dB)')
plt.tight_layout(); plt.show()

## Exercises

1. **CT filters:** add a cosine roll-off to the ramp (`|w| * cos`) and compare noise vs sharpness against pure Ram-Lak on a noisy sinogram.
2. **CT iterative:** implement gradient descent on \(\tfrac12\norm{Gx-y}^2+\lambda\,\mathrm{TV}(x)\) with `radon` as \(G\) and `back_project(...,ramp=False)` as \(G^{\top}\); beat FBP at 8 angles.
3. **CT limited-angle:** restrict `thetas` to [0,120) degrees and describe the artefacts (this is assignment a2's incomplete-data problem).
4. **MRI rate:** sweep the random under-sampling fraction 0.5 → 0.1; when does anatomy become unreadable? Does keeping the k-space centre matter?
5. **MRI + prior:** denoise the under-sampled MRI reconstruction with `week_1.5_to_3` Notebook 2's Huber MAP denoiser — a poor-man's compressed sensing.

**Where this goes:** with sparsity priors these are exactly compressed-sensing
CT/MRI; the deep-learning weeks replace the hand-built prior with a learned one.